# **Notebook 4: RAG Implementation**
## Assignment: Hybrid RAG & Fine-Tuning for Customer Support
---

### TO-DO: Before Running This Notebook

**Files you NEED:**
- [ ] `corporate_policies/` folder with `.md` SOP files
- [ ] `outputs.json` — Created by Notebook 3
- [ ] GPU runtime enabled

**Files this notebook will CREATE:**
- [ ] `./chroma_db/` — Persisted ChromaDB vector index _(Required by NB5 and NB7)_
- [ ] `outputs.json` (updated) — adds `naive_rag_output` _(Required by NB5 and NB7)_

---

### **Task 3.2: Implement Retrieval-Assisted Generation**

#### **3.2.1 Generate Embeddings [4 marks]**
**The Task:** Initialise the `all-MiniLM-L6-v2` embedding model, embed the SOP documents, and validate the embeddings were produced.

**Hints & Tips:**
* Load the SOP documents with `TextLoader` first (or reuse the corpus from NB2).
* `HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")` runs on CPU — no GPU needed.
* Validate by embedding one sample string and checking the vector length (384 dims for MiniLM).
* You MUST use the same embedding model when reloading in Notebooks 5 and 7.

**Embedding Model Options:**
* **`all-MiniLM-L6-v2`** (recommended): 384-dim, fast, ~80MB.
* **`all-mpnet-base-v2`**: 768-dim, higher quality, slower.
* **`bge-small-en-v1.5`**: 384-dim, newer architecture.

**Learner Inference:** Your text is now coordinates in semantic space — similar meanings sit close together.

In [1]:
%pip install -q -U bitsandbytes accelerate langchain-community langchain-huggingface langchain-chroma langchain-text-splitters chromadb sentence-transformers

import os, glob, json, shutil
import numpy as np
import pandas as pd
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings

from google.colab import drive
if not os.path.isdir('/content/drive/MyDrive'):
    drive.mount('/content/drive')
os.chdir('/content/drive/MyDrive/hybrid_rag')

with open('outputs.json') as f:
    outputs = json.load(f)
test_query = outputs['test_query']

# same loading and chunking as NB2
policy_files = sorted(glob.glob('corporate_policies/*.md'))
docs = [d for f in policy_files for d in TextLoader(f, encoding='utf-8').load()]
splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=0, separators=['\n## ', '\n\n', '\n', ' '])
chunks = splitter.split_documents(docs)
for d in chunks:
    d.metadata['sop'] = os.path.basename(d.metadata['source'])
print(len(chunks), 'chunks')

EMBED_MODEL = 'sentence-transformers/all-MiniLM-L6-v2'
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL, encode_kwargs={'normalize_embeddings': True})

vectors = np.array(embeddings.embed_documents([d.page_content for d in chunks]))
query_vec = np.array(embeddings.embed_query(test_query))
print('Document vectors:', vectors.shape, '| query vector:', query_vec.shape)

assert vectors.shape == (len(chunks), 384)
assert np.isfinite(vectors).all()
assert np.allclose(np.linalg.norm(vectors, axis=1), 1, atol=1e-3)

sims = vectors @ vectors.T
np.fill_diagonal(sims, np.nan)
print(f'Cosine similarity between different SOPs: mean {np.nanmean(sims):.3f}, max {np.nanmax(sims):.3f}')

np.save('sop_embeddings.npy', vectors)

/tmp/ipykernel_26915/3007911125.py:6: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


13 chunks


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Document vectors: (13, 384) | query vector: (384,)
Cosine similarity between different SOPs: mean 0.389, max 0.774


#### **3.2.2 Build Vector Index [4 marks]**
**The Task:** Create a persistent Chroma vector index from the embedded documents, configure similarity search, and validate the index.

**Hints & Tips:**
* `Chroma.from_documents(docs, embeddings, persist_directory="./chroma_db")` auto-saves — no manual `.persist()` needed.
* Validate with `vector_db._collection.count()` — should equal the number of SOP documents.
* Run one test `.similarity_search("refund", k=1)` to confirm retrieval works.

**Vector DB Options:**
* **ChromaDB** (recommended): simple API, auto-persistence, LangChain integration.
* **FAISS**: faster for >100K docs, but no built-in persistence (manual serialization).

**Learner Inference:** The index lets you search by meaning — it returns the document mathematically closest to your query's coordinates.

In [2]:
from langchain_chroma import Chroma

# Chroma's SQLite file doesn't work well directly on the Drive mount, so build it in /tmp and copy it over
LOCAL_DB = '/tmp/chroma_db'
shutil.rmtree(LOCAL_DB, ignore_errors=True)
vector_db = Chroma.from_documents(chunks, embeddings, collection_name='sop_policies', persist_directory=LOCAL_DB,
                                  collection_metadata={'hnsw:space': 'cosine'})

print('Vectors in index:', vector_db._collection.count())
assert vector_db._collection.count() == len(chunks)
print('"refund" ->', vector_db.similarity_search('refund', k=1)[0].metadata['sop'])

# the V2 router will search with these strings, so each one has to bring back its own SOP
with open('intent_sop_map.json') as f:
    intent_sop_map = json.load(f)
check = []
for sop in sorted({v['sop_file'] for v in intent_sop_map.values()}):
    query = next(v['search_string'] for v in intent_sop_map.values() if v['sop_file'] == sop)
    doc, dist = vector_db.similarity_search_with_score(query, k=1)[0]
    check.append({'sop': sop, 'retrieved': doc.metadata['sop'], 'distance': round(dist, 3)})
check = pd.DataFrame(check)
display(check)
print(f"{(check.sop == check.retrieved).sum()} / {len(check)} search strings find their SOP")

shutil.rmtree('chroma_db', ignore_errors=True)
shutil.copytree(LOCAL_DB, 'chroma_db')

Vectors in index: 13
"refund" -> refund_policy.md


,sop,retrieved,distance
0,account_recovery.md,account_recovery.md,-0.0
1,billing_disputes.md,billing_disputes.md,-0.0
2,data_privacy.md,data_privacy.md,-0.0
3,escalation_matrix.md,escalation_matrix.md,0.0
4,order_tracking.md,order_tracking.md,0.0
5,password_reset.md,password_reset.md,0.0
6,payment_methods.md,payment_methods.md,-0.0
7,refund_policy.md,refund_policy.md,-0.0
8,shipping_delays.md,shipping_delays.md,0.0
9,subscription_cancellation.md,subscription_cancellation.md,-0.0


12 / 12 search strings find their SOP


'chroma_db'

#### **3.2.3 Implement Retrieval Workflow [4 marks]**
**The Task:** Execute a "Naive RAG" workflow — pass the raw customer query into the vector DB, fetch the top result, and augment the LLM prompt.

**Hints & Tips:**
* Use `.similarity_search(query, k=1)` for the top-1 document.
* Check whether the raw query retrieved the WRONG policy — common with ambiguous queries.
* Inject context via the system prompt: `"Answer strictly using this SOP: {context}"`.

**Parameter Tuning:**
* `k=1`: one document (focused). `k=3`: more context if SOPs overlap. `k=5`: max, risks long prompts.

**Learner Inference:** Noisy queries often retrieve the wrong document — proving Naive RAG is flawed and motivating the fine-tuned router.

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = outputs['model_id']
GEN_KWARGS = outputs['gen_kwargs']

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                                bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'left'
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map='auto')
model.eval()

@torch.no_grad()
def generate(messages_list, batch_size=16):
    replies = []
    for i in range(0, len(messages_list), batch_size):
        prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True)
                   for m in messages_list[i:i + batch_size]]
        batch = tokenizer(prompts, return_tensors='pt', padding=True).to(model.device)
        out = model.generate(**batch, **GEN_KWARGS, pad_token_id=tokenizer.pad_token_id)
        replies += tokenizer.batch_decode(out[:, batch['input_ids'].shape[1]:], skip_special_tokens=True)
    return [r.strip() for r in replies]

TOP_K = 1   # one whole policy is enough context; more just mixes in other rules
RAG_PROMPT = (outputs['baseline_prompt'] + '\nAnswer strictly using this SOP:\n\n{context}\n\n'
              'If the SOP does not cover the question, say so and offer to escalate to a human agent.')

def rag_messages(query, context):
    return [{'role': 'system', 'content': RAG_PROMPT.format(context=context)}, {'role': 'user', 'content': query}]

# naive RAG: search with the raw customer message
hits = vector_db.similarity_search_with_score(test_query, k=TOP_K)
retrieved_sop = hits[0][0].metadata['sop']
print('Query:', test_query)
print(f'Retrieved: {retrieved_sop} (distance {hits[0][1]:.3f}), expected shipping_delays.md')

context = '\n\n'.join(doc.page_content for doc, _ in hits)
naive_rag_output = generate([rag_messages(test_query, context)])[0]

print('\nBaseline:', outputs['baseline_output'])
print('\nNaive RAG:', naive_rag_output)
print('\nGround truth:', outputs['ground_truth'])

# a few messy messages to see how raw-text search copes
noisy = [
    ('oh great, charged AGAIN for something i cancelled. love it.', 'billing_disputes.md'),
    ('cant get in, says too many attempts, did someone hack me??', 'account_recovery.md'),
    ('i want to talk to an actual human not a bot', 'escalation_matrix.md'),
    ('so when exactly are you people open', 'working_hours.md'),
]
print()
for q, expected in noisy:
    got = vector_db.similarity_search(q, k=1)[0].metadata['sop']
    print(f'{"ok   " if got == expected else "wrong"} {q} -> {got}')

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Query: ugh my order STILL hasn't arrived... how many days is delivery even supposed to take??
Retrieved: shipping_delays.md (distance 0.423), expected shipping_delays.md



Baseline: I'm sorry you're experiencing this delay. Typically, orders should arrive within 3-5 business days after being placed. Please check your shipping confirmation or contact our customer service team for more information on when your order is expected to be delivered. We appreciate your patience as we work to resolve any issues.

Naive RAG: The estimated delivery time for your order was between 10-21 business days. Given the typical timeline, it seems like your order might still be delayed. Please check your tracking information regularly and keep me updated if you need further assistance.

Ground truth: Domestic orders deliver within 3–7 business days; international orders within 10–21 business days. An order is only considered "delayed" once it has passed the upper bound of its quoted window. Carrier scans can lag reality by up to 24 hours, so a single stale tracking event is not by itself a delay.

wrong oh great, charged AGAIN for something i cancelled. love it. -> subscript

---
## Save Artifacts for Downstream Notebooks

In [4]:
outputs['naive_rag_output'] = naive_rag_output
outputs['naive_rag_retrieved_sop'] = retrieved_sop
outputs['rag_config'] = {'embed_model': EMBED_MODEL, 'collection': 'sop_policies', 'top_k': TOP_K,
                         'n_chunks': len(chunks), 'rag_prompt': RAG_PROMPT}
with open('outputs.json', 'w') as f:
    json.dump(outputs, f, indent=2)
print(os.listdir('chroma_db'))

['0f508e2a-4823-46d1-aa71-0919d4d5022b', 'chroma.sqlite3']


---
## END-OF-NOTEBOOK CHECKLIST

> **IMPORTANT: Verify before proceeding to Notebook 5.**

- [ ] SOP documents loaded via `TextLoader`
- [ ] **Embeddings generated and validated** ← _Task 3.2.1_
- [ ] **ChromaDB index built, validated, and persisted** ← _Task 3.2.2_
- [ ] Naive similarity search executed on `test_query`
- [ ] Naive RAG output generated with SOP context injected
- [ ] **`./chroma_db/` exists on disk** ← _CRITICAL for NB5 and NB7_
- [ ] **`outputs.json` updated** with `naive_rag_output` ← _CRITICAL for NB5 and NB7_

**If any item is unchecked, fix it before moving on.**